<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-16/notebooks/ClimatePipeline/03_Climate_PresionAtmosferica_DailyProcessor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate_PresionAtmosferica_DailyProcessor

Transforma una partición mensual climática subdiaria en una capa diaria por `estación + sensor + día`. Actualmente admite precipitación y la familia de temperatura (ambiente, mínima y máxima) mediante contratos separados.

## Garantías de esta etapa

- Los Parquet de `clima_crudo` son de solo lectura.
- Los sensores paralelos permanecen separados.
- Los duplicados se eliminan con trazabilidad.
- Los conflictos se excluyen del agregado y se exportan para revisión.
- La cadencia se infiere por estación, sensor y partición mensual.
- No se imputan observaciones ni se eliminan extremos.
- Cada variable usa su contrato: precipitación se suma; temperatura conserva media, mediana, mínimo y máximo sin sumarse.
- El valor diario aceptado permanece en `NaN` hasta aprobar la regla de cobertura en la auditoría diaria.
- El manifiesto `COMPLETA` se guarda solo al terminar todas las salidas.

Esta etapa no crea todavía el calendario completo, no decide cobertura mínima y no escoge el sensor canónico de una estación.


## 1. Preparar el repositorio en Colab

Colab abre el notebook desde GitHub, pero no descarga automáticamente los módulos `.py`. Esta celda clona el repositorio en `/content`. Durante desarrollo usa `feature/SCRUM-16`; antes de la entrega final se fijará un commit concreto.


In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-16'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})


## 2. Configuración y plan

El ejemplo predeterminado conserva **precipitación, Cundinamarca, febrero de 2025**. Para temperatura se deben cambiar conjuntamente `VARIABLE_NOMBRE` y `DATASET_ID` según la tabla de contratos. Todas las banderas quedan protegidas para que `Run all` no procese datos por accidente.

| Variable | Dataset | Estadístico diario principal | Estado |
|---|---|---|---|
| `precipitacion` | `s54a-sgyg` | suma | Validado |
| `temperatura_ambiente` | `sbwg-7ju4` | media | Piloto pendiente |
| `temperatura_minima` | `afdg-3zpb` | mínimo | Piloto pendiente |
| `temperatura_maxima` | `ccvq-rp9s` | máximo | Piloto pendiente |
| `humedad` | `uext-mhny` | por definir | ⚠️ Bloqueado |
| `presion_atmosferica` | `62tk-nxj5` | por definir | ⚠️ Bloqueado |
| `velocidad_viento` | `sgfv-3yp8` | por definir | ⚠️ Bloqueado |

Para paralelizar, cada persona debe usar particiones distintas y un `WORKER_ID` identificable.


In [ ]:
import json
import time

import pandas as pd

from ClimateProcessingUtils import (
    ahora_proyecto,
    construir_plan_particiones,
    detectar_commit,
    descubrir_partes_parquet,
    escribir_json_atomico,
    escribir_parquet_atomico,
    formatear_duracion,
    inventariar_parquets,
    leer_particion_parquet,
    ruta_particion_cruda,
    ruta_particion_diaria,
)
from DatasetConfig import cargar_configuracion_datasets
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

VARIABLE_NOMBRE = 'presion_atmosferica'
DATASET_ID = '62tk-nxj5'
PROCESAR_DEPARTAMENTOS = ['CUNDINAMARCA', 'BOYACÁ']
PROCESAR_ANIOS = [2024, 2025]
PROCESAR_MESES = list(range(1, 13))

WORKER_ID = 'worker_a'
MAX_PARTICIONES = None  # Procesa incrementalmente las 48 particiones.
SOBRESCRIBIR_RESULTADOS = False
EJECUTAR_PROCESAMIENTO = False  # Banderita de seguridad para Run all.

DATASET_CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)
PROCESSED_ROOT = DATASET_CONFIG.processed_root

if VARIABLE_NOMBRE == 'precipitacion':
    from PrecipitationRules import (
        COLUMNAS_REQUERIDAS,
        RULE_VERSION,
        procesar_precipitacion as PROCESAR_VARIABLE,
    )
elif VARIABLE_NOMBRE in {
    'temperatura_ambiente',
    'temperatura_minima',
    'temperatura_maxima',
}:
    from TemperatureRules import (
        COLUMNAS_REQUERIDAS,
        RULE_VERSION,
        procesar_temperatura as PROCESAR_VARIABLE,
    )
elif VARIABLE_NOMBRE == 'humedad':
    from HumidityRules import detener_contrato_pendiente

    detener_contrato_pendiente()
elif VARIABLE_NOMBRE == 'presion_atmosferica':
    from AtmosphericPressureRules import (
        COLUMNAS_REQUERIDAS,
        RULE_VERSION,
        procesar_presion_atmosferica as PROCESAR_VARIABLE,
    )
elif VARIABLE_NOMBRE == 'velocidad_viento':
    from WindSpeedRules import detener_contrato_pendiente

    detener_contrato_pendiente()
else:
    raise ValueError(
        'Variable sin contrato diario. Ejecute 02 y cree reglas específicas antes de usar 03.'
    )
if not str(WORKER_ID).strip():
    raise ValueError('WORKER_ID es obligatorio para la trazabilidad.')
if MAX_PARTICIONES is not None and int(MAX_PARTICIONES) <= 0:
    raise ValueError('MAX_PARTICIONES debe ser None o un entero positivo.')

plan_particiones = construir_plan_particiones(
    VARIABLE_NOMBRE,
    DATASET_ID,
    PROCESAR_DEPARTAMENTOS,
    PROCESAR_ANIOS,
    PROCESAR_MESES,
)
if MAX_PARTICIONES is not None:
    plan_particiones = plan_particiones[: int(MAX_PARTICIONES)]

print({
    'regla_version': RULE_VERSION,
    'worker_id': WORKER_ID,
    'particiones': len(plan_particiones),
    'sobrescribir': SOBRESCRIBIR_RESULTADOS,
    'ejecutar': EJECUTAR_PROCESAMIENTO,
    'processed_root': str(PROCESSED_ROOT),
})


In [ ]:
plan_df = pd.DataFrame([
    {
        **spec.como_dict(),
        'entrada': str(ruta_particion_cruda(PROCESSED_ROOT, spec)),
        'salida': str(ruta_particion_diaria(PROCESSED_ROOT, spec)),
    }
    for spec in plan_particiones
])
display(Markdown(f'### Plan configurado: {len(plan_df)} partición(es)'))
display(plan_df)


## 3. Procesamiento de una partición

Cada partición genera `observaciones_diarias`, `cadencias`, `duplicados_eliminados`, `conflictos`, `rechazados`, `resumen_procesamiento` y `manifest.json`.

Una salida con manifiesto `COMPLETA` se omite cuando la sobrescritura está desactivada. Una carpeta incompleta exige revisión explícita.


In [ ]:
NOMBRES_SALIDA = {
    'diario': 'observaciones_diarias.parquet',
    'cadencias': 'cadencias.parquet',
    'duplicados': 'duplicados_eliminados.parquet',
    'conflictos': 'conflictos.parquet',
    'rechazados': 'rechazados.parquet',
    'resumen': 'resumen_procesamiento.parquet',
    'manifest': 'manifest.json',
}


def leer_manifiesto_completo(output_dir):
    ruta = output_dir / NOMBRES_SALIDA['manifest']
    if not ruta.exists():
        return None
    contenido = json.loads(ruta.read_text(encoding='utf-8'))
    return contenido if contenido.get('estado') == 'COMPLETA' else None


def validar_resultado(resultado, filas_entrada):
    if resultado.metricas['filas_entrada'] != filas_entrada:
        raise RuntimeError('El procesador no registró todas las filas de entrada.')
    if filas_entrada and resultado.diario.empty:
        raise RuntimeError('La partición cruda no produjo observaciones diarias.')
    if not resultado.diario.empty:
        claves = ['codigoestacion', 'codigosensor', 'fecha']
        if resultado.diario.duplicated(claves).any():
            raise RuntimeError('La salida diaria contiene claves duplicadas.')
        if VARIABLE_NOMBRE == 'precipitacion':
            if resultado.diario['precipitacion_observada_mm'].lt(0).any():
                raise RuntimeError('La salida diaria contiene precipitación negativa.')
        elif VARIABLE_NOMBRE.startswith('temperatura_'):
            principal = resultado.diario['temperatura_principal_observada_c']
            if principal.isna().any():
                raise RuntimeError('La salida diaria contiene temperatura principal nula.')
        else:
            principal = resultado.diario['valor_principal_observado']
            if principal.isna().any():
                raise RuntimeError('La salida diaria contiene valor principal nulo.')


def resumen_existente(spec, manifiesto, output_dir):
    return {
        **spec.como_dict(),
        'estado': 'omitida_ya_completa',
        'filas_entrada': manifiesto.get('entrada', {}).get('filas'),
        'filas_diarias_salida': manifiesto.get('metricas', {}).get('filas_diarias_salida'),
        'duracion_segundos': 0.0,
        'salida': str(output_dir),
    }


def procesar_particion(spec):
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    input_dir = ruta_particion_cruda(PROCESSED_ROOT, spec)
    output_dir = ruta_particion_diaria(PROCESSED_ROOT, spec)

    manifiesto_existente = leer_manifiesto_completo(output_dir)
    if manifiesto_existente and not SOBRESCRIBIR_RESULTADOS:
        print(f'Ya está completa; se omite: {output_dir}')
        return resumen_existente(spec, manifiesto_existente, output_dir)
    if output_dir.exists() and any(output_dir.iterdir()) and not SOBRESCRIBIR_RESULTADOS:
        raise RuntimeError(f'Existe una salida incompleta: {output_dir}')

    manifest_path = output_dir / NOMBRES_SALIDA['manifest']
    escribir_json_atomico(
        {
            'estado': 'INICIADA',
            'regla_version': RULE_VERSION,
            'worker_id': WORKER_ID,
            'commit': detectar_commit(REPO_DIR),
            'particion': spec.como_dict(),
            'inicio': inicio.isoformat(),
        },
        manifest_path,
        sobrescribir=SOBRESCRIBIR_RESULTADOS,
    )

    archivos = descubrir_partes_parquet(input_dir)
    inventario = inventariar_parquets(archivos)
    filas_inventario = int(inventario['filas'].sum())
    tamano_entrada = int(inventario['tamano_bytes'].sum())
    print(f'Leyendo {len(archivos):,} archivos y {filas_inventario:,} filas.')
    crudo = leer_particion_parquet(archivos, columnas=COLUMNAS_REQUERIDAS)
    if len(crudo) != filas_inventario:
        raise RuntimeError(f'Filas leídas ({len(crudo):,}) != metadatos ({filas_inventario:,}).')

    resultado = PROCESAR_VARIABLE(crudo, spec)
    validar_resultado(resultado, filas_inventario)
    tablas = {
        'diario': resultado.diario,
        'cadencias': resultado.cadencias,
        'duplicados': resultado.duplicados_eliminados,
        'conflictos': resultado.conflictos,
        'rechazados': resultado.rechazados,
    }
    rutas_salida = {}
    for nombre, tabla in tablas.items():
        ruta = output_dir / NOMBRES_SALIDA[nombre]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_RESULTADOS)
        rutas_salida[nombre] = {
            'ruta': str(ruta),
            'filas': len(tabla),
            'tamano_bytes': ruta.stat().st_size,
        }

    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    resumen = {
        **spec.como_dict(),
        **resultado.metricas,
        'worker_id': WORKER_ID,
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'duracion_legible': formatear_duracion(duracion),
        'archivos_entrada': len(archivos),
        'tamano_entrada_bytes': tamano_entrada,
        'tamano_salida_bytes': sum(item['tamano_bytes'] for item in rutas_salida.values()),
        'salida': str(output_dir),
    }
    ruta_resumen = output_dir / NOMBRES_SALIDA['resumen']
    tabla_resumen = pd.DataFrame([resumen])
    escribir_parquet_atomico(
        tabla_resumen,
        ruta_resumen,
        sobrescribir=SOBRESCRIBIR_RESULTADOS,
    )
    rutas_salida['resumen'] = {
        'ruta': str(ruta_resumen),
        'filas': len(tabla_resumen),
        'tamano_bytes': ruta_resumen.stat().st_size,
    }

    manifiesto = {
        'estado': 'COMPLETA',
        'regla_version': RULE_VERSION,
        'worker_id': WORKER_ID,
        'commit': detectar_commit(REPO_DIR),
        'particion': spec.como_dict(),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'entrada': {
            'ruta': str(input_dir),
            'archivos': len(archivos),
            'filas': filas_inventario,
            'tamano_bytes': tamano_entrada,
        },
        'metricas': resultado.metricas,
        'salidas': rutas_salida,
    }
    escribir_json_atomico(
        manifiesto,
        manifest_path,
        sobrescribir=True,
    )
    print(f'Partición completa: {len(resultado.diario):,} filas diarias | {formatear_duracion(duracion)}')
    return {'estado': 'completa', **resumen}


def registrar_error(spec, exc):
    output_dir = ruta_particion_diaria(PROCESSED_ROOT, spec)
    contenido = {
        'estado': 'ERROR',
        'worker_id': WORKER_ID,
        'regla_version': RULE_VERSION,
        'commit': detectar_commit(REPO_DIR),
        'particion': spec.como_dict(),
        'momento': ahora_proyecto().isoformat(),
        'error_tipo': type(exc).__name__,
        'error_mensaje': str(exc),
    }
    escribir_json_atomico(
        contenido,
        output_dir / f'manifest_error_{WORKER_ID}.json',
        sobrescribir=True,
    )
    return {
        **spec.como_dict(),
        'estado': 'error',
        'error': f'{type(exc).__name__}: {exc}',
        'salida': str(output_dir),
    }


## 4. Ejecución protegida

Revise el plan con `EJECUTAR_PROCESAMIENTO=False`. Después cambie la bandera a `True`, ejecute nuevamente la celda de configuración para actualizar el valor en memoria y finalmente ejecute esta celda protegida. Si una partición falla, se registra el error y la corrida continúa.


In [ ]:
resumen_corrida = pd.DataFrame()

if not EJECUTAR_PROCESAMIENTO:
    print('Procesamiento desactivado. Revise el plan y active EJECUTAR_PROCESAMIENTO.')
else:
    resumenes = []
    inicio_corrida = time.perf_counter()
    for numero, spec in enumerate(plan_particiones, start=1):
        display(Markdown(
            f'---\n\n## Partición {numero}/{len(plan_particiones)}: '
            f'{spec.departamento} {spec.anio}-{spec.mes:02d}'
        ))
        try:
            resumenes.append(procesar_particion(spec))
        except Exception as exc:
            print(f'ERROR: {type(exc).__name__}: {exc}')
            resumenes.append(registrar_error(spec, exc))

    resumen_corrida = pd.DataFrame(resumenes)
    display(Markdown('---\n\n## Resumen de la corrida'))
    display(resumen_corrida)
    print(f'Duración total: {formatear_duracion(time.perf_counter() - inicio_corrida)}')


## 5. Interpretación y siguiente paso

`cobertura_observada_pct` usa la cadencia modal del par estación-sensor durante el mes. Puede superar 100 % si la frecuencia cambia y todavía no declara que un día sea completo.

Después del piloto deben revisarse agregados, pares sin cadencia, coberturas extremas, conflictos, rechazados, sensores paralelos, tiempo y tamaño. Cada familia de variables necesita reglas propias también en `04` y `05`; no se debe escalar el histórico usando la auditoría o consolidación de otra variable.
